# 04 - Test the step 9 agent gateway (agent.usp_record_decision) against the private ERP/WMS Azure SQL database

Exercises AA-01 (dedup), AA-02 (owners), AA-04 (staging), and the Tier 2 gateway rules (HL-03/04/05/07/11, idempotency,
blocked-retry, guardrail, materiality + Finance co-approval). Also proves the agent identity cannot write ERP/WMS tables
directly but can through the procedures (ownership chaining). Uses trigger_id `TEST` and **always cleans up** (reserved
units restored, ERP rows deleted). All data is **SYNTHETIC**.

In [ ]:
dbutils.widgets.text("sql_server", "", "a. SQL server FQDN")
dbutils.widgets.text("sql_database", "sqldb-appareliq-erpwms", "b. SQL database")
dbutils.widgets.text("service_credential", "appareliq-sql", "c. UC service credential")
dbutils.widgets.text("agent_principal", "id-appareliq-agent", "d. agent SQL user")

In [ ]:
import json, uuid, datetime
server, database = dbutils.widgets.get("sql_server"), dbutils.widgets.get("sql_database")
agent_user = dbutils.widgets.get("agent_principal")
assert server, "set the sql_server widget"
token = dbutils.credentials.getServiceCredentialsProvider(dbutils.widgets.get("service_credential")).get_token("https://database.windows.net/.default").token
url = (f"jdbc:sqlserver://{server}:1433;database={database};encrypt=true;trustServerCertificate=false;"
       "hostNameInCertificate=*.database.windows.net;loginTimeout=60")
props = spark._jvm.java.util.Properties(); props.setProperty("accessToken", token)
conn = spark._jvm.java.sql.DriverManager.getConnection(url, props)
results, cases = [], []

def q(sql, *params):
    ps = conn.prepareStatement(sql)
    for i, p in enumerate(params, 1):
        if p is None: ps.setNull(i, 12)
        elif isinstance(p, int): ps.setInt(i, p)
        elif isinstance(p, float): ps.setBigDecimal(i, spark._jvm.java.math.BigDecimal(str(p)))
        else: ps.setString(i, str(p))
    has = ps.execute(); rows = []
    while True:
        if has:
            rs = ps.getResultSet(); md = rs.getMetaData(); n = md.getColumnCount()
            cols = [md.getColumnLabel(i + 1) for i in range(n)]; rows = []
            while rs.next():
                rows.append({cols[i]: (None if rs.getObject(i + 1) is None else str(rs.getObject(i + 1))) for i in range(n)})
        elif ps.getUpdateCount() == -1:
            break
        has = ps.getMoreResults()
    return rows

def check(name, cond, detail=""):
    results.append({"test": name, "pass": bool(cond), "detail": str(detail)[:300]})
    print(("PASS " if cond else "FAIL ") + name, str(detail)[:200])

def decide(action_id, case_id, decision, upn, option_id=None, units=None, rationale=None, reason=None, defer=None, mat=50000.0):
    return q("EXEC agent.usp_record_decision @action_id=?, @case_id=?, @decision=?, @approver_upn=?, @option_id=?, @units=?, "
             "@rationale=?, @reason_code=?, @defer_until_utc=?, @materiality_gm_usd=?",
             action_id, case_id, decision, upn, option_id, units, rationale, reason, defer, mat)[0]

def reserved(loc, sku):
    return int(q("SELECT ISNULL(reserved_units,0) r FROM wms.inventory_position WHERE location_id=? AND sku_id=?", loc, sku)[0]["r"])

fallback = q("SELECT upn FROM agent.persona_user WHERE is_escalation = 1")[0]["upn"]
fx = q("""
SELECT TOP 1 s.style_id, s.category, s.lifecycle, k.sku_id, sl.location_id AS src, sl.market_id AS src_mkt,
       dl.location_id AS dst, dl.market_id AS dst_mkt, t.mode, t.lead_days, t.cost_per_unit_usd AS lane_cost,
       CAST(p.full_price_local / m.fx_usd_to_local AS DECIMAL(19,4)) AS price, s.unit_cost_usd AS ucost,
       g.max_transfer_cost_pct AS cap, pu.upn AS owner_upn, o.approval_limit_units AS lim,
       ISNULL(i.on_hand_units,0) - ISNULL(i.allocated_units,0) - ISNULL(i.reserved_units,0) AS avail
FROM erp.product_style s
JOIN erp.product_sku k ON k.style_id = s.style_id
JOIN wms.inventory_position i ON i.sku_id = k.sku_id
JOIN erp.location sl ON sl.location_id = i.location_id AND sl.location_type = 'DC'
JOIN wms.transfer_lane t ON t.from_location_id = sl.location_id
JOIN erp.location dl ON dl.location_id = t.to_location_id AND dl.location_type = 'DC' AND dl.market_id <> sl.market_id
JOIN erp.price p ON p.style_id = s.style_id AND p.market_id = dl.market_id
JOIN ref.market m ON m.market_id = dl.market_id
JOIN erp.margin_guardrail g ON g.market_id = dl.market_id AND g.category = s.category AND g.lifecycle = s.lifecycle
JOIN ref.decision_owner o ON o.category = s.category AND o.market_id = dl.market_id AND o.role = 'ACCOUNTABLE'
JOIN agent.persona_user pu ON pu.person_id = o.person_id AND pu.upn <> ?
WHERE ISNULL(i.on_hand_units,0) - ISNULL(i.allocated_units,0) - ISNULL(i.reserved_units,0) >= 60
  AND t.cost_per_unit_usd / (p.full_price_local / m.fx_usd_to_local) <= g.max_transfer_cost_pct
  AND NOT EXISTS (SELECT 1 FROM agent.[case] c WHERE c.style_id = s.style_id AND c.market_id = dl.market_id)
  AND (SELECT COUNT(*) FROM ref.decision_owner o2 WHERE o2.category = s.category AND o2.market_id = dl.market_id AND o2.role = 'ACCOUNTABLE') = 1
ORDER BY avail DESC, s.style_id""", fallback)
assert fx, "no fixture found"
fx = fx[0]; print(json.dumps(fx, indent=1))
fin = q("""SELECT TOP 1 pu.upn FROM ref.decision_owner o JOIN agent.persona_user pu ON pu.person_id = o.person_id
           WHERE o.role = 'FINANCE' AND o.category = ? AND o.market_id = ?""", fx["category"], fx["dst_mkt"])[0]["upn"]
outsider = q("""SELECT TOP 1 upn FROM agent.persona_user WHERE upn NOT IN (?, ?, ?) AND upn NOT IN (
                  SELECT pu.upn FROM ref.decision_owner o JOIN agent.persona_user pu ON pu.person_id = o.person_id
                  WHERE o.category = ? AND o.market_id = ? AND o.role IN ('ACCOUNTABLE','FINANCE'))""",
             fx["owner_upn"], fin, fallback, fx["category"], fx["dst_mkt"])[0]["upn"]
owner, src, sku = fx["owner_upn"], fx["src"], fx["sku_id"]
line = lambda u: json.dumps([{"sku_id": sku, "from_location_id": src, "to_location_id": fx["dst"], "units": u,
                              "lane_mode": fx["mode"], "lead_days": int(fx["lead_days"]), "cost_per_unit_usd": float(fx["lane_cost"]),
                              "full_price_usd": float(fx["price"]), "unit_cost_usd": float(fx["ucost"])}])
rec = json.dumps({"recommended_option_id": "S1", "options": [
    {"option_id": "S0", "label": "Maintain allocation", "units": 0},
    {"option_id": "S1", "label": f"Transfer from {src}", "units": 30}], "data_label": "SYNTHETIC"})
ev = json.dumps({"sku_id": sku, "test": True, "data_label": "SYNTHETIC"})
base_reserved = reserved(src, sku)
run = uuid.uuid4().hex[:8]
try:
    # ---- AA-01 open + dedup
    c1 = q("EXEC agent.usp_open_case ?, ?, ?, ?, ?, ?, ?, ?, ?", "TEST", fx["style_id"], fx["dst_mkt"], "HIGH",
           "SYNTHETIC test case", ev, rec, "S1", "test-harness")[0]
    case1 = c1["case_id"]; cases.append(case1)
    check("AA-01 open new case", c1["is_new"] in ("1", "true", "True"), c1)
    c1b = q("EXEC agent.usp_open_case ?, ?, ?, ?, ?, ?, ?, ?, ?", "TEST", fx["style_id"], fx["dst_mkt"], "HIGH",
            "SYNTHETIC test case (repeat)", ev, rec, "S1", "test-harness")[0]
    check("AA-01 dedup same style x market", c1b["case_id"] == case1 and c1b["is_new"] in ("0", "false", "False"), c1b)
    # ---- AA-04 stage
    st = q("EXEC agent.usp_stage_transfer_draft ?, ?, ?, ?", case1, "S1", line(30), "test-harness")[0]
    check("AA-04 stage draft", st["lines_staged"] == "1", st)
    check("AA-04 staging reserves nothing", reserved(src, sku) == base_reserved)
    # ---- AA-02 owners
    owners = q("EXEC agent.usp_get_case_owners ?", case1)
    roles = {o["case_role"] for o in owners}
    check("AA-02 owners include ACCOUNTABLE, FINANCE, SOURCE_ACCOUNTABLE",
          {"ACCOUNTABLE", "FINANCE", "SOURCE_ACCOUNTABLE"} <= roles, sorted(roles))
    check("AA-02 accountable owner first", owners[0]["case_role"] == "ACCOUNTABLE" and owners[0]["upn"] == owner, owners[0])
    q("EXEC agent.usp_set_case_chat ?, ?, ?", case1, f"19:test-{run}@thread.v2", None)
    # ---- HL-11 non-owner
    r = decide(f"t{run}-a1", case1, "APPROVE", outsider, "S1")
    check("HL-11 non-owner cannot decide", r["outcome"] == "BLOCKED" and "accountable owner" in r["message"], r["message"])
    # ---- HL-03 scope then corrected retry with same action id
    r = decide(f"t{run}-a2", case1, "REVISE", owner, "S1", 45, "more")
    check("HL-03 units above staged scope blocked", r["outcome"] == "BLOCKED" and "between 1 and the staged 30" in r["message"], r["message"])
    r = decide(f"t{run}-a2", case1, "REVISE", owner, "S1", 20)
    check("HL-03 revise without rationale blocked", r["outcome"] == "BLOCKED" and "rationale" in r["message"], r["message"])
    r = decide(f"t{run}-a2", case1, "REVISE", owner, "S1", 20, "Keep 10 units at source for local event (SYNTHETIC test)")
    check("Gateway executes revise (blocked attempts do not consume the key)", r["outcome"] == "EXECUTED" and r["approved_units"] == "20", r)
    check("Reserved units at source increased by 20", reserved(src, sku) == base_reserved + 20, reserved(src, sku))
    sto = q("SELECT COUNT(*) n, SUM(units) u, MIN(sto_status) s FROM erp.stock_transfer_order WHERE sto_id = ?", r["sto_id"])[0]
    check("STO created with 20 units, status CREATED", sto["n"] == "1" and sto["u"] == "20" and sto["s"] == "CREATED", sto)
    dec = q("SELECT decision, approved_by_upn, JSON_VALUE(evidence_json,'$.case.case_id') cid, JSON_VALUE(evidence_json,'$.chosen_option_id') opt, "
            "JSON_VALUE(evidence_json,'$.approved_units') au, JSON_QUERY(evidence_json,'$.options') opts FROM erp.allocation_decision WHERE proposal_id = ?", case1)
    check("ERP decision MODIFIED with HIST-shaped evidence", len(dec) == 1 and dec[0]["decision"] == "MODIFIED" and dec[0]["cid"] == case1
          and dec[0]["opt"] == "S1" and dec[0]["au"] == "20" and dec[0]["opts"], dec)
    fu = q("SELECT STRING_AGG(task_type, ',') WITHIN GROUP (ORDER BY task_type) t FROM agent.follow_up WHERE case_id = ?", case1)[0]["t"]
    check("AA-05 follow-ups T+7 / T+14 scheduled", fu == "CHECK_T14,CHECK_T7", fu)
    dr = q("SELECT draft_status, released_units FROM agent.transfer_draft WHERE case_id = ?", case1)[0]
    check("Draft RELEASED with released_units", dr["draft_status"] == "RELEASED" and dr["released_units"] == "20", dr)
    # ---- idempotency + closed case
    r = decide(f"t{run}-a2", case1, "REVISE", owner, "S1", 20, "retry")
    check("Idempotent replay returns DUPLICATE", r["outcome"] == "DUPLICATE" and r["approved_units"] == "20", r["message"])
    check("Replay did not double-reserve", reserved(src, sku) == base_reserved + 20)
    r = decide(f"t{run}-a3", case1, "REJECT", owner)
    check("Closed case accepts no further decisions", r["outcome"] == "BLOCKED" and "APPROVED" in r["message"], r["message"])

    # ---- HL-05 materiality: owner approve -> NEEDS_FINANCE -> Finance co-approves
    c2 = q("EXEC agent.usp_open_case ?, ?, ?, ?, ?, ?, ?, ?, ?", "TEST", fx["style_id"], fx["dst_mkt"], "HIGH",
           "SYNTHETIC test case 2", ev, rec, "S1", "test-harness")[0]
    case2 = c2["case_id"]; cases.append(case2)
    check("AA-01 new case after the previous one closed", case2 != case1 and case2.endswith("-02"), case2)
    q("EXEC agent.usp_stage_transfer_draft ?, ?, ?, ?", case2, "S1", line(15), "test-harness")
    r = decide(f"t{run}-b1", case2, "APPROVE", owner, "S1", None, None, None, None, 1.0)
    check("HL-05 above materiality -> PENDING_FINANCE", r["outcome"] == "PENDING_FINANCE" and r["case_status"] == "NEEDS_FINANCE", r["message"])
    check("PENDING_FINANCE writes nothing to ERP/WMS", reserved(src, sku) == base_reserved + 20 and
          q("SELECT COUNT(*) n FROM erp.allocation_decision WHERE proposal_id = ?", case2)[0]["n"] == "0")
    r = decide(f"t{run}-b2", case2, "APPROVE", owner)
    check("HL-05 owner cannot self co-approve", r["outcome"] == "BLOCKED" and "Finance" in r["message"], r["message"])
    r = decide(f"t{run}-b3", case2, "APPROVE", fin)
    check("HL-05 Finance co-approval executes pending decision", r["outcome"] == "EXECUTED" and r["approved_units"] == "15", r)
    rat = q("SELECT approved_by_upn, rationale, JSON_VALUE(evidence_json,'$.gateway.finance_co_approver_upn') co FROM erp.allocation_decision WHERE proposal_id = ?", case2)[0]
    check("Decision attributed to owner with Finance co-approver", rat["approved_by_upn"] == owner and rat["co"] == fin, rat)
    check("Reserved units now +35", reserved(src, sku) == base_reserved + 35)

    # ---- HL-04 defer + HL-07 reject on case 3
    c3 = q("EXEC agent.usp_open_case ?, ?, ?, ?, ?, ?, ?, ?, ?", "TEST", fx["style_id"], fx["dst_mkt"], "MEDIUM",
           "SYNTHETIC test case 3", ev, rec, "S1", "test-harness")[0]
    case3 = c3["case_id"]; cases.append(case3)
    past = (datetime.datetime.utcnow() - datetime.timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")
    future = (datetime.datetime.utcnow() + datetime.timedelta(days=3)).strftime("%Y-%m-%d %H:%M:%S")
    r = decide(f"t{run}-c1", case3, "DEFER", owner, None, None, None, None, past)
    check("HL-04 defer to a past date blocked", r["outcome"] == "BLOCKED", r["message"])
    r = decide(f"t{run}-c1", case3, "DEFER", owner, None, None, None, None, future)
    check("HL-04 defer to a future date", r["outcome"] == "DEFERRED" and r["case_status"] == "DEFERRED", r["message"])
    check("HL-04 re-open follow-up scheduled",
          q("SELECT COUNT(*) n FROM agent.follow_up WHERE case_id = ? AND task_type = 'REOPEN_DEFERRED'", case3)[0]["n"] == "1")
    c3b = q("EXEC agent.usp_open_case ?, ?, ?, ?, ?, ?, ?, ?, ?", "TEST", fx["style_id"], fx["dst_mkt"], "HIGH",
            "SYNTHETIC repeat while deferred", ev, rec, "S1", "test-harness")[0]
    check("AA-01 repeat trigger during an active deferral is deduplicated", c3b["case_id"] == case3 and c3b["case_status"] == "DEFERRED", c3b)
    q("EXEC agent.usp_stage_transfer_draft ?, ?, ?, ?", case3, "S1", line(10), "test-harness")
    r = decide(f"t{run}-c2", case3, "OVERRIDE", owner, "S0", None, "Hold for launch", None)
    check("HL-06 override without reason code blocked", r["outcome"] == "BLOCKED" and "reason code" in r["message"], r["message"])
    r = decide(f"t{run}-c3", case3, "REJECT", owner, None, None, "Not needed (SYNTHETIC test)")
    check("HL-07 reject recorded", r["outcome"] == "REJECTED" and r["case_status"] == "REJECTED", r["message"])
    check("Reject cancels drafts and reserves nothing", reserved(src, sku) == base_reserved + 35 and
          q("SELECT COUNT(*) n FROM agent.transfer_draft WHERE case_id = ? AND draft_status = 'PROPOSED'", case3)[0]["n"] == "0")

    # ---- CW-07 re-simulation replaces the recommendation and all drafts atomically (twice: no PK clash with CANCELLED lines)
    c4 = q("EXEC agent.usp_open_case ?, ?, ?, ?, ?, ?, ?, ?, ?", "TEST", fx["style_id"], fx["dst_mkt"], "HIGH",
           "SYNTHETIC test case 4", ev, rec, "S1", "test-harness")[0]
    case4 = c4["case_id"]; cases.append(case4)
    q("EXEC agent.usp_stage_transfer_draft ?, ?, ?, ?", case4, "S1", line(10), "test-harness")
    rec2 = json.dumps({"recommended_option_id": "S3", "options": [], "data_label": "SYNTHETIC"})
    for n in (8, 6):
        drafts = json.dumps([{"option_id": "S1", "lines": json.loads(line(n))}, {"option_id": "S3", "lines": json.loads(line(4))}])
        q("EXEC agent.usp_update_recommendation @case_id=?, @recommendation_json=?, @recommended_option_id=?, @drafts_json=?, @actor=?, @detail_json=?",
          case4, rec2, "S3", drafts, "test-harness", json.dumps({"constraint": {"max_units": n}}))
    d4 = q("SELECT option_id, SUM(units) u FROM agent.transfer_draft WHERE case_id = ? AND draft_status = 'PROPOSED' GROUP BY option_id ORDER BY option_id", case4)
    check("CW-07 re-simulation replaces drafts (repeatable)", [(x["option_id"], x["u"]) for x in d4] == [("S1", "6"), ("S3", "4")], d4)
    rc = q("SELECT recommended_option_id r FROM agent.[case] WHERE case_id = ?", case4)[0]["r"]
    n7 = q("SELECT COUNT(*) n FROM agent.action_log WHERE case_id = ? AND action_type = 'CW-07_RESIMULATE'", case4)[0]["n"]
    check("CW-07 recommendation updated and logged", rc == "S3" and n7 == "2", f"{rc} {n7}")
    q("EXEC agent.usp_stage_transfer_draft ?, ?, ?, ?", case4, "S1", line(5), "test-harness")
    check("Re-stage after re-simulation (no PK clash)",
          q("SELECT SUM(units) u FROM agent.transfer_draft WHERE case_id = ? AND option_id = 'S1' AND draft_status = 'PROPOSED'", case4)[0]["u"] == "5")
    r = decide(f"t{run}-d1", case4, "REJECT", owner, None, None, "SYNTHETIC test")
    check("Case 4 closed", r["outcome"] == "REJECTED", r["message"])

    # ---- once-only markers (reminders, signal updates) rely on usp_log_action returning inserted 1 then 0
    m1 = q("EXEC agent.usp_log_action ?, ?, ?, ?, ?, ?, ?", f"t{run}-mark", case4, 1, "AA-06_SLA_REMIND", "test-harness", "DONE", None)[0]
    m2 = q("EXEC agent.usp_log_action ?, ?, ?, ?, ?, ?, ?", f"t{run}-mark", case4, 1, "AA-06_SLA_REMIND", "test-harness", "DONE", None)[0]
    check("usp_log_action is once-only (1 then 0)", m1["inserted"] == "1" and m2["inserted"] == "0", f"{m1} {m2}")
    mem = [q("EXEC agent.usp_add_case_member ?, ?, ?, ?", case4, "P-TEST", owner, "PLANNER")[0]["inserted"] for _ in range(2)]
    check("usp_add_case_member is idempotent (1 then 0)", mem == ["1", "0"], mem)

    # ---- Least privilege: agent identity
    st_ = conn.createStatement()
    st_.execute(f"EXECUTE AS USER = N'{agent_user}'")
    try:
        try:
            st_.execute("UPDATE wms.inventory_position SET reserved_units = reserved_units WHERE 1 = 0")
            check("Agent identity cannot UPDATE wms directly", False, "update succeeded")
        except Exception as e:
            check("Agent identity cannot UPDATE wms directly", "denied" in str(e).lower(), str(e)[:160])
        try:
            st_.execute("INSERT INTO erp.allocation_decision (decision_id) VALUES (NEWID())")
            check("Agent identity cannot INSERT erp directly", False, "insert succeeded")
        except Exception as e:
            check("Agent identity cannot INSERT erp directly", "denied" in str(e).lower(), str(e)[:160])
        rs = st_.executeQuery(f"EXEC agent.usp_get_case N'{case1}'"); rs.next()
        check("Agent identity can read a case through the procedures", case1 in (rs.getString(1) or ""))
    finally:
        st_.execute("REVERT")
finally:
    # ---- cleanup (always)
    for cid in cases:
        q("""UPDATE i SET reserved_units = i.reserved_units - d.released_units
             FROM wms.inventory_position i JOIN agent.transfer_draft d ON d.from_location_id = i.location_id AND d.sku_id = i.sku_id
             WHERE d.case_id = ? AND d.draft_status = 'RELEASED' AND d.released_units > 0""", cid)
        q("DELETE FROM erp.stock_transfer_order WHERE decision_id IN (SELECT decision_id FROM erp.allocation_decision WHERE proposal_id = ?)", cid)
        q("DELETE FROM erp.allocation_decision WHERE proposal_id = ?", cid)
        for t in ("follow_up", "case_member", "transfer_draft", "action_log"):
            q(f"DELETE FROM agent.{t} WHERE case_id = ?", cid)
        q("DELETE FROM agent.action_log WHERE action_id LIKE ?", f"t{run}-%")
        q("DELETE FROM agent.[case] WHERE case_id = ?", cid)
    restored = reserved(src, sku) == base_reserved
    left = q("SELECT COUNT(*) n FROM agent.[case] WHERE trigger_id = 'TEST'")[0]["n"]
    conn.close()
check("Cleanup restored reserved units and removed TEST cases", restored and left == "0", f"left={left}")
failed = [r for r in results if not r["pass"]]
print(f"{len(results) - len(failed)}/{len(results)} passed")
dbutils.notebook.exit(json.dumps({"passed": len(results) - len(failed), "total": len(results), "failed": failed, "fixture": fx}))